In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
TEXT_DATA_PATH = "D:/data/proceeded/meld_features_updated.csv"
data = pd.read_csv(TEXT_DATA_PATH)

# Drop rows where 'Clean_Utterance' is NaN
data = data.dropna(subset=['Clean_Utterance'])

# Extract text (utterances) and responses (clean utterances)
X_text = data['Utterance'].astype(str).tolist()  # Input utterances
y_responses = data['Clean_Utterance'].astype(str).tolist()  # Target responses

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_text, y_responses, test_size=0.2, random_state=42)

print(f"Training Set Size: {len(X_train)}, Test Set Size: {len(X_test)}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model and tokenizer details
model_name = "meta-llama/Llama-2-7b-chat-hf"
access_token = ""  # Replace with your Hugging Face token

# Quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=getattr(torch, "float16"),
    bnb_4bit_use_double_quant=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=access_token)

# # #Load Llama-2 model with quantization
# model = AutoModelForCausalLM.from_pretrained(
#      model_name,
#      quantization_config=quant_config,
#      device_map="auto",
#      use_auth_token=access_token,
#      load_in_8bit_fp32_cpu_offload=True
# )
model = AutoModelForCausalLM.from_pretrained(model_name, 
  quantization_config=quant_config, 
  device_map={"":0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print("Llama-2 Model Loaded Successfully!")

In [ ]:
# Define the prompt template
instruction = "Generate a response to the following utterance."
prompt_template = "<s>[INST] <<SYS>>{instruction}<</SYS>>{utterance}[/INST]{response}</s>"

# Prepare the dataset
train_data = [prompt_template.format(instruction=instruction, utterance=text, response=response) 
              for text, response in zip(X_train, y_train)]
test_data = [prompt_template.format(instruction=instruction, utterance=text, response=response) 
             for text, response in zip(X_test, y_test)]

from datasets import Dataset
# Convert lists to dictionaries
train_data_dict = {"text": train_data}
test_data_dict = {"text": test_data}

# Create Hugging Face Dataset objects
train_dataset = Dataset.from_dict(train_data_dict)
test_dataset = Dataset.from_dict(test_data_dict)


# Print a sample to verify
print("Sample Train Data:", train_data[0])
print("Sample Test Data:", test_data[0])

# Set the padding token
tokenizer.pad_token = tokenizer.eos_token 

def tokenize_function(examples):
    # Tokenize the input text
    tokenized = tokenizer(
        examples["text"],  # Input text
        padding="max_length",  # Pad to max_length
        truncation=True,  # Truncate to max_length
        max_length=512,  # Set max_length
        return_tensors="pt",  # Return PyTorch tensors
    )
    
    # Add labels (same as input_ids for causal language modeling)
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

# Tokenize train and test data
train_encodings = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
test_encodings = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# # Print tokenized output for verification
# print("Train Encodings Input IDs Shape:", train_encodings['input_ids'].shape)
# print("Train Encodings Attention Mask Shape:", train_encodings['attention_mask'].shape)
print("Dataset Prepared Successfully!")

In [ ]:
from peft import LoraConfig, get_peft_model

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank of the low-rank adaptation
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Target layers for LoRA
    lora_dropout=0.1,  # Dropout for LoRA layers
    bias="none",  # No bias for LoRA
    task_type="CAUSAL_LM",  # Task type
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
print("LoRA Applied to the Model Successfully!")

In [ ]:
# Move the model to the GPU (if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Verify the model is on the correct device
print(f"Model is on device: {next(model.parameters()).device}")

In [ ]:
from transformers import TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir="./llama2-finetuned",  # Directory to save the fine-tuned model
    per_device_train_batch_size=2,  # Batch size for training
    per_device_eval_batch_size=2,  # Batch size for evaluation
    num_train_epochs=3,  # Number of training epochs
    logging_dir="./logs",  # Directory to store logs
    logging_steps=10,  # Log every 10 steps
    save_steps=500,  # Save model every 500 steps
    evaluation_strategy="steps",  # Evaluate every `eval_steps`
    eval_steps=500,  # Evaluate every 500 steps
    save_total_limit=2,  # Keep only the last 2 saved models
    fp16=False,  # Use mixed precision (FP16)
)

print("Training Arguments Defined Successfully!")

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encodings,
    eval_dataset=test_encodings
)


# Fine-tune the model
trainer.train()

print("Fine-Tuning Completed Successfully!")